# KoCharELECTRA + BIO v2 — Resume Training (+5 epoch fine-tune)

**Resume from**: kocharelectra_bio_v2_best.pt (val 0.5767 at epoch 5, val loss 계속 떨어지는 추세 — 학습 더 필요)
**전략**: LR 1e-5로 낮춰서 5 epoch 더 fine-tune
**저장**: kocharelectra_bio_v2_resume_best.pt

## 업로드 파일
1. `bio_train_v2.jsonl` (학습 데이터)
2. `kocharelectra_bio_v2_best.pt` (이전 best checkpoint)

In [ ]:
!pip install -q transformers==4.46.0 accelerate
import sys, os, json, time, random
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    print(f'  uploaded: {name} ({len(uploaded[name])/1024:.0f}KB)')

In [ ]:
# Load data
records = []
with open('bio_train_v2.jsonl', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'Total records: {len(records)}')
print(f'Total chars: {sum(r["n_chars"] for r in records):,}')
print(f'Total B-SENT: {sum(r["n_matched"] for r in records):,}')
print(f'avg match rate: {sum(r["match_rate"] for r in records)/len(records)*100:.1f}%')
print(f'sample text[:200]: {records[0]["text"][:200]}')

In [ ]:
# Train/val split
random.seed(42)
random.shuffle(records)
n_val = max(50, len(records) // 10)
val_records = records[:n_val]
train_records = records[n_val:]
print(f'Train: {len(train_records)}, Val: {len(val_records)}')

In [ ]:
from transformers import AutoTokenizer, ElectraModel
MODEL_ID = 'monologg/kocharelectra-small-discriminator'
MAX_LEN = 512
STRIDE = 384  # sliding window stride (overlap = 128 chars)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f'Tokenizer vocab size: {tokenizer.vocab_size}')
print(f'Sample tokenization: {tokenizer.tokenize("학부모님 안녕하십니까?")}')

In [ ]:
# Dataset with sliding window (긴 문서 처리)
class BIODataset(Dataset):
    """각 PDF text를 sliding window로 chunk화. 각 chunk를 학습 sample.

    char-level 라벨을 token-level로 변환:
      tokenizer가 char_to_token 매핑 제공 (return_offsets_mapping=True)
      각 token의 첫 char 라벨 사용
    """
    def __init__(self, records, tokenizer, max_len=512, stride=384):
        self.samples = []
        for rec in records:
            text = rec['text']
            char_labels = rec['labels']  # list[int]
            # Tokenize with offsets — char position → token position
            enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False, truncation=False)
            tokens = enc['input_ids']
            offsets = enc['offset_mapping']
            # Token-level label: 각 token의 첫 char 라벨
            tok_labels = []
            for start, end in offsets:
                if start < len(char_labels):
                    tok_labels.append(char_labels[start])
                else:
                    tok_labels.append(0)  # O
            # Sliding window
            for i in range(0, len(tokens), stride):
                chunk_tokens = tokens[i:i + max_len - 2]  # CLS + SEP 자리
                chunk_labels = tok_labels[i:i + max_len - 2]
                if len(chunk_tokens) < 8:
                    continue
                # CLS + tokens + SEP, labels는 CLS/SEP 자리에 -100 (ignore)
                input_ids = [tokenizer.cls_token_id] + chunk_tokens + [tokenizer.sep_token_id]
                labels = [-100] + chunk_labels + [-100]
                # Pad to max_len
                attn_mask = [1] * len(input_ids)
                pad_len = max_len - len(input_ids)
                input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
                attn_mask = attn_mask + [0] * pad_len
                labels = labels + [-100] * pad_len
                self.samples.append({
                    'input_ids': torch.tensor(input_ids, dtype=torch.long),
                    'attention_mask': torch.tensor(attn_mask, dtype=torch.long),
                    'labels': torch.tensor(labels, dtype=torch.long),
                })
                if i + max_len - 2 >= len(tokens):
                    break
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

print('Building train dataset (sliding window)...')
train_ds = BIODataset(train_records, tokenizer, MAX_LEN, STRIDE)
print(f'Train samples (chunks): {len(train_ds)}')
val_ds = BIODataset(val_records, tokenizer, MAX_LEN, STRIDE)
print(f'Val samples (chunks): {len(val_ds)}')

BATCH = 16
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}')

In [ ]:
# Model: KoCharELECTRA + BIO classifier — Resume from checkpoint
class BIOTagger(nn.Module):
    def __init__(self, model_id=MODEL_ID, num_labels=3, dropout=0.1):
        super().__init__()
        self.electra = ElectraModel.from_pretrained(model_id)
        hidden = self.electra.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)
    def forward(self, input_ids, attention_mask, labels=None):
        out = self.electra(input_ids=input_ids, attention_mask=attention_mask)
        hidden = self.dropout(out.last_hidden_state)
        logits = self.classifier(hidden)  # (B, T, 3)
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100,
                                          weight=torch.tensor([1.0, 5.0, 1.0]).to(logits.device))
            loss = loss_fn(logits.view(-1, 3), labels.view(-1))
        return {'loss': loss, 'logits': logits}

model = BIOTagger().to(device)
# === Resume: 기존 best checkpoint 로드 ===
ckpt = torch.load('kocharelectra_bio_v2_best.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded checkpoint: model_id={ckpt.get("model_id")}, max_len={ckpt.get("max_len")}')

n_params = sum(p.numel() for p in model.parameters())
print(f'Total: {n_params/1e6:.1f}M')

# LR 낮춤 — fine-tune
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
EPOCHS = 5
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# Training loop — resume + 5 epoch
best_val = float('inf')
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    t0 = time.time()
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        out = model(input_ids, attn, labels)
        loss = out['loss']
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    model.eval()
    val_loss = 0.0
    correct_B = 0; total_B = 0
    correct_I = 0; total_I = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attn = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            out = model(input_ids, attn, labels)
            val_loss += out['loss'].item()
            preds = out['logits'].argmax(dim=-1)
            mask = labels != -100
            for cls, name in [(1, 'B'), (2, 'I')]:
                cls_mask = (labels == cls) & mask
                correct = ((preds == cls) & cls_mask).sum().item()
                total = cls_mask.sum().item()
                if cls == 1:
                    correct_B += correct; total_B += total
                else:
                    correct_I += correct; total_I += total
    train_loss /= max(len(train_loader), 1)
    val_loss /= max(len(val_loader), 1)
    b_acc = correct_B / max(total_B, 1)
    i_acc = correct_I / max(total_I, 1)
    elapsed = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  B-acc={b_acc:.3f} I-acc={i_acc:.3f}  ({elapsed:.0f}s)')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'state_dict': model.state_dict(), 'model_id': MODEL_ID, 'max_len': MAX_LEN}, 'kocharelectra_bio_v2_resume_best.pt')
        print('  > saved best')

print(f'Best val: {best_val:.4f}')

In [ ]:
# Quick test on val sample
import json
model.eval()
sample = val_records[0]
text = sample['text'][:400]
enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False, truncation=True, max_length=510)
input_ids = torch.tensor([tokenizer.cls_token_id] + enc['input_ids'] + [tokenizer.sep_token_id]).unsqueeze(0).to(device)
attn = torch.ones_like(input_ids).to(device)
with torch.no_grad():
    out = model(input_ids, attn)
preds = out['logits'].argmax(dim=-1)[0].cpu().tolist()

# CLS 빼고
preds = preds[1:1 + len(enc['input_ids'])]
# B 위치 찾기 → sentence boundary
print(f'Text ({len(text)} chars):')
print(text)
print()
print('Predicted sentences:')
current_start = None
for tok_i, (start, end) in enumerate(enc['offset_mapping']):
    pred = preds[tok_i]
    if pred == 1:  # B-SENT
        if current_start is not None:
            print(f'  {text[current_start:start]}')
        current_start = start
if current_start is not None:
    print(f'  {text[current_start:]}')

In [ ]:
from google.colab import files
files.download('kocharelectra_bio_v2_resume_best.pt')